In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from shapely import wkb
import geopandas as gpd
import pyodbc
from sqlalchemy import create_engine
from sqlalchemy import text   
from sqlalchemy import event
import urllib
import os
from thefuzz import fuzz
from thefuzz import process

In [2]:
print(pyodbc.drivers())

['SQL Server', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)', 'ODBC Driver 18 for SQL Server']


In [3]:
SERVER = "MMLDAPP03"
SOURCE_DB = "MMLDGIS"           
PRIMARY_WIRES_TABLE = "PRIMARY_CONDUCTOR" 
SECONDARY_WIRES_TABLE = "SECONDARY_CONDUCTOR"
TABLES = [PRIMARY_WIRES_TABLE, SECONDARY_WIRES_TABLE]

In [4]:
# Connect to MS SQL database
def connect_to_db(server, database):
    # 1. Verify exact driver name available to python instance
    drivers = pyodbc.drivers()
    print(f"Available drivers: {drivers}")
    
    # Select Driver 18 explicitly (most recent as of 2024-07-07)
    driver_name = "ODBC Driver 18 for SQL Server"
    
    if driver_name not in drivers:
        raise Exception(f"Driver '{driver_name}' not found. Available: {drivers}")

    # 2. Construct RAW string (NO urllib.parse.quote_plus)
    conn_str = (
        f"DRIVER={{{driver_name}}};"
        f"SERVER={server};"
        f"DATABASE={database};"
        f"Trusted_Connection=yes;"
        f"MARS_Connection=yes;"
        f"Encrypt=yes;"
        f"TrustServerCertificate=yes;" 
    )
    
    print(f"Attempting connection with: {conn_str}")
    
    try:
        # 3. Connect directly
        conn = pyodbc.connect(conn_str)
        print("Database connection established successfully!")
        return conn
    except Exception as e:
        print(f"Database connection failed!\n{e}")

In [5]:
# Connect to the database using pyodbc
conn = connect_to_db(SERVER, SOURCE_DB)

Available drivers: ['SQL Server', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)', 'ODBC Driver 18 for SQL Server']
Attempting connection with: DRIVER={ODBC Driver 18 for SQL Server};SERVER=MMLDAPP03;DATABASE=MMLDGIS;Trusted_Connection=yes;MARS_Connection=yes;Encrypt=yes;TrustServerCertificate=yes;
Database connection established successfully!


In [10]:
def sql_column_exists(cursor, table_name, column_name):
    t_name = table_name.replace('[', '').replace(']', '')
    c_name = column_name.replace('[', '').replace(']', '')
    
    cursor.execute("""
        SELECT COUNT(*) 
        FROM INFORMATION_SCHEMA.COLUMNS 
        WHERE TABLE_SCHEMA = 'dbo' 
          AND TABLE_NAME = ? 
          AND COLUMN_NAME = ?
    """, (t_name, c_name))
    
    return cursor.fetchone()[0] > 0   

In [8]:
query = "SELECT DISTINCT PHASE FROM [dbo].[{table}]"
pd.read_sql(query.format(table=PRIMARY_WIRES_TABLE), conn)

C:\Users\bknight\AppData\Local\Temp\ipykernel_26760\251017827.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(query.format(table=PRIMARY_WIRES_TABLE), conn)


,PHASE
0,THREE
1,NaN
2,SINGLE


In [11]:
base_columns = [
    "[SHAPE].STAsBinary() as shape_wkb",
    "[ENABLED]",
    "[OH_UG]",
    "[SUBTYPECODE]",
    "[FEEDERID]",
    "[OPERVOLT]",
    "[LOCATIONID]",
    "[LENGTH]",
    "[WIRESIZE]",
    "[GISID]",
    "[PARENTID]",
    "[UPDN_PROBLEM]",
    "[NETWORK_PROBLEM]"
]

optional_columns = ["[PHASE]"]

all_gdfs = []
cursor = conn.cursor()

for table in TABLES:
    print(f"Processing {table}...")

    # Build SELECT query
    select_list = base_columns.copy()

    for col in optional_columns:
        if sql_column_exists(cursor, table, col):
            select_list.append(col)
            print(f"{col} exists in {table}!")
        else:
            select_list.append(f"NULL AS {col}")

    # Construct the final query
    final_query = f"""
        SELECT {', '.join(select_list)}
        FROM [dbo].[{table}]
    """
    # Execute and load
    df = pd.read_sql(final_query, conn)

    if "PHASE" in df.columns: 
        print("Unique phases:")
        print(df["PHASE"].unique())

    # Convert WKB binary to Shapely geometry
    df["geometry"] = df["shape_wkb"].apply(lambda x: wkb.loads(x))

    # Create GeoDataFrame
    gdf = gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:2249")

    all_gdfs.append(gdf)
    
print(select_list)
# Access results
primary_wires = all_gdfs[0]
primary_wires["SOURCETABLE"] = "PRIMARY"
secondary_wires = all_gdfs[1]
secondary_wires["SOURCETABLE"] = "SECONDARY"

print("Success!")

Processing PRIMARY_CONDUCTOR...
[PHASE] exists in PRIMARY_CONDUCTOR!
Unique phases:
<StringArray>
['SINGLE', 'THREE', nan]
Length: 3, dtype: str
Processing SECONDARY_CONDUCTOR...
Unique phases:
[None]
['[SHAPE].STAsBinary() as shape_wkb', '[ENABLED]', '[OH_UG]', '[SUBTYPECODE]', '[FEEDERID]', '[OPERVOLT]', '[LOCATIONID]', '[LENGTH]', '[WIRESIZE]', '[GISID]', '[PARENTID]', '[UPDN_PROBLEM]', '[NETWORK_PROBLEM]', 'NULL AS [PHASE]']
Success!


C:\Users\bknight\AppData\Local\Temp\ipykernel_26760\909150947.py:41: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(final_query, conn)


In [12]:
# Append both dataframes together
print("Merging wire tables...")
wires = pd.concat([primary_wires, secondary_wires], ignore_index=True)
print("Success!")
display(wires.head())

Merging wire tables...
Success!


,shape_wkb,ENABLED,OH_UG,SUBTYPECODE,FEEDERID,OPERVOLT,LOCATIONID,LENGTH,WIRESIZE,GISID,PARENTID,UPDN_PROBLEM,NETWORK_PROBLEM,PHASE,geometry,SOURCETABLE
0,b'\x01\x02\x00\x00\x00\x02\x00\x00\x00\x00\xb3...,1.0,OH,1.0,TEDESCO,80.0,Glendale Rd,9.031308,2/0 CU Strand,PL00001,PL02103,CHILDLESS,DOWNSTREAM_ONLY,SINGLE,"LINESTRING (824117.666 3003201.359, 824112.042...",PRIMARY
1,b'\x01\x02\x00\x00\x00\x02\x00\x00\x00\x00\xb3...,1.0,OH,3.0,TEDESCO,110.0,Glendale Rd,658.000807,3/0 AL Spacer Cable,PL00002,PL02103,GOOD,GOOD,THREE,"LINESTRING (824117.666 3003201.359, 823615.264...",PRIMARY
2,b'\x01\x02\x00\x00\x00\x02\x00\x00\x00\x00K\x1...,1.0,OH,1.0,TEDESCO,80.0,Glendale Rd,8.078712,2/0 CU Strand,PL00003,PL00002,CHILDLESS,DOWNSTREAM_ONLY,SINGLE,"LINESTRING (823615.264 3003626.279, 823607.219...",PRIMARY
3,b'\x01\x02\x00\x00\x00\x03\x00\x00\x00\x00K\x1...,1.0,OH,3.0,TEDESCO,110.0,Glendale Rd,369.999110,3/0 AL Spacer Cable,PL00004,PL00002,GOOD,GOOD,THREE,"LINESTRING (823615.264 3003626.279, 823408.815...",PRIMARY
4,b'\x01\x02\x00\x00\x00\x02\x00\x00\x00\x80\x98...,1.0,OH,3.0,TEDESCO,110.0,Humphrey St,67.941158,3/0 AL Spacer Cable,PL00005,SW00111,GOOD,DOWNSTREAM_ONLY,THREE,"LINESTRING (823365.378 3003890.839, 823421.253...",PRIMARY


In [13]:
primary_wires["PHASE"].unique()

<StringArray>
['SINGLE', 'THREE', nan]
Length: 3, dtype: str

In [14]:
cable_cm_map = {
    "#1 CU EPR": 83690,        # 1 AWG
    "#2 AL Spacer Cable": 66360,  # 2 AWG
    "#2 AL URD": 66360,        # 2 AWG
    "#2 CU Package Cable": 66360, # 2 AWG
    "#2 CU Strand": 66360,     # 2 AWG
    "#3 AL Spacer Cable": 52620,  # 3 AWG
    "#4 5Kv": 41740,           # 4 AWG (Voltage rating does not affect conductor area)
    "#4 CU Solid": 41740,      # 4 AWG
    "#6 CU Solid": 26240,      # 6 AWG
    "1/0 Aerial Tree Wire": 105600, # 1/0 AWG
    "1/0 CU Strand": 105600,   # 1/0 AWG
    "2/0 AL Triplex": 133100,  # 2/0 AWG
    "2/0 CU Strand": 133100,   # 2/0 AWG
    "3/0 AL Spacer Cable": 167800, # 3/0 AWG
    "336 AL Spacer Cable": 336000, # 336 MCM * 1,000
    "500 MCM CU": 500000,      # 500 MCM * 1,000
    "795 AL Spacer Cable": 795000, # 795 MCM * 1,000
    "Spacer Cable": None       # Size undefined without gauge or MCM specification
}   

In [ ]:
# Calculate current
def get_current(wire, v_drop_pct, formula="standard"):
    """ 
    Function to calculate an approximate amperage per a single row (wire) given wire size,
    length, operating voltage, and a standard voltage drop percentage.

    V_drop = OPERVOLT * (%Drop/100)
    I_single = (V_drop * CM)/(2 * K * L)
    I_three = (V_drop * CM)/(1.732 * K * L)

    v_drop_pct: Estimated voltage drop across a single wire in decimal form
    I: Maximum current in Amps
    V_drop: Allowable voltage drop in volts calculated from %
    CM: Circular Mils of wire size
    K: Resistivity constant of the material (Cu=12.9, Al=21.2)
    L: One-way length of the wire run in feet
    Single-Phase and Three-Phase Constants: 2 or 1.732 for single and three, respectively
    """

    # Get values
    v_drop = float(wire["OPERVOLT"]) * (v_drop_pct)
    size = str(wire["WIRESIZE"]).strip()
    phase = str(wire["PHASE"])
    length = float(wire["LENGTH"]) # double check units

    cm_map_value = cable_cm_map.get(size)
    circular_mils = float(cable_cm_map.get(size)) if cm_map_value else None

    # Determine resistance based on wire material
    print(size)
    if size and "CU" in size:
        resistance = 12.9
    elif size and "AL" in size:
        resistance = 21.2
    else:
        return None # Unknown material or invalid size string
    
    if circular_mils is not None and length > 0:
        if phase and "SINGLE" in phase:
            return (v_drop * circular_mils)/(2 * resistance * length)
        elif phase and "THREE" in phase:
            return (v_drop * circular_mils)/(1.732 * resistance * length)
        else:
            return None
        

In [ ]:
# Onderdonk & Preece's Equations
# 1. Preece (simpler)
# I = A * d^(3/2); 
# A = constant of material; 
# d = diameter of wire; 
# I = current

# 2. Onderdonk (more complex)
# I = A * sqrt((log_10(T_melt-T_ambient)/(234-T_ambient) + 1)/(33 * t)); 
# A = cross-sectional area in circular mils; 
# t = time in seconds the current is applied; 
# delta_T = the rise in temp from the initial state; 
# Ta = reference temperature in C

# Steady-State Thermal Balance Equation
# I^2 * R * T_c + q_s = q_c + q_r
# I = current
# R = resistance (increases as the wire gets hotter)
# q_s = heat gained from the sun (solar heating)
# q_c = heat lost to the wind (convective cooling)
# q_r = heat lost via thermal radiation

In [ ]:
import math

def get_overhead_failure_current(wire, ambient_f=95, wind_speed_mph=2.0, max_temp_c=90):
    """
    Estimates the current that will raise an overhead wire to max_temp_c 
    under specific ambient temperature and wind conditions.
    Based on simplified IEEE 738 thermal balance principles.
    
    :param wire: Pandas row containing 'WIRESIZE' and material info
    :param ambient_f: Ambient air temperature in Fahrenheit (e.g., 95 for hot day)
    :param wind_speed_mph: Wind speed in mph (Critical for overhead cooling; 0-5 mph typical)
    :param max_temp_c: Maximum allowable conductor temperature (e.g., 75 for safe, 150+ for failure)
    """
    
    # 1. Get Base Properties
    size = str(wire["WIRESIZE"])
    raw_cm = cable_cm_map.get(size)
    if raw_cm is None: return None
    
    cm = float(raw_cm)
    diameter_inches = math.sqrt(cm) / 1000 * 1.15 # Approx diameter for stranded wire
    
    # Material Limits & Resistivity
    if "CU" in size:
        alpha_20 = 0.00393  # Temp coeff of resistance
        r_20_per_ft = (10.4 / cm) / 1000 # Ohms per ft at 20C (approx)
        melt_temp = 1083
    elif "AL" in size:
        alpha_20 = 0.00403
        r_20_per_ft = (17.0 / cm) / 1000
        melt_temp = 660
    else:
        return None

    # 2. Convert Units
    ambient_c = (ambient_f - 32) * 5/9
    wind_m_s = wind_speed_mph * 0.44704 # Convert to m/s
    
    # 3. Calculate Resistance at Max Temp
    # R(T) = R(20) * [1 + alpha * (T - 20)]
    r_max = r_20_per_ft * (1 + alpha_20 * (max_temp_c - 20))
    
    # 4. Estimate Heat Loss (Cooling) - Simplified IEEE 738
    # Convection (Forced) ~ sqrt(wind_speed) * (T_cond - T_amb)
    # Radiation ~ (T_cond^4 - T_amb^4)
    # Note: This is a simplified approximation.
    
    delta_t = max_temp_c - ambient_c
    if delta_t <= 0: return 0 # Ambient is already hotter than limit
    
    # Simplified Cooling Power (Watts/ft) - Calibrated for typical overhead conditions
    # Convection dominates in wind; Radiation dominates in still air
    q_conv = 0.65 * math.sqrt(wind_m_s) * delta_t * (diameter_inches * 0.0254) # Approx convective coeff
    q_rad = 0.015 * ((max_temp_c + 273)**4 - (ambient_c + 273)**4) * (diameter_inches * 0.0254) * 1e-8 # Approx radiative
    
    # Solar Gain (Watts/ft) - Adds heat, reducing capacity
    # Typical summer sun: 90-100 W/sqft projected area
    q_solar = 12.0 * (diameter_inches * 0.0254) # Approx solar gain per ft
    
    total_cooling = q_conv + q_rad
    net_cooling = total_cooling - q_solar # Net heat dissipation available for current
    
    if net_cooling <= 0:
        return 0 # Solar gain exceeds cooling; wire will heat up even with 0 current
        
    # 5. Solve for Current: I = sqrt(Net_Cooling / Resistance)
    # Note: Resistance is per foot, Cooling is per foot. Units cancel to Amps.
    i_limit = math.sqrt(net_cooling / r_max)
    
    return i_limit   

In [16]:
test = wires["WIRESIZE"].iloc[2]
print(test)
if "CU" in test:
    print("yes")

2/0 CU Strand
yes


In [24]:
v_drop_pct = 0.06
wires["CALC_CURRENT"] = wires.apply(get_current, axis=1, v_drop_pct=v_drop_pct)

2/0 CU Strand
3/0 AL Spacer Cable
2/0 CU Strand
3/0 AL Spacer Cable
3/0 AL Spacer Cable
3/0 AL Spacer Cable
3/0 AL Spacer Cable
3/0 AL Spacer Cable
#2 CU Strand
3/0 AL Spacer Cable
3/0 AL Spacer Cable
#2 CU Strand
3/0 AL Spacer Cable
3/0 AL Spacer Cable
2/0 CU Strand
3/0 AL Spacer Cable
2/0 CU Strand
3/0 AL Spacer Cable
2/0 CU Strand
3/0 AL Spacer Cable
#4 CU Solid
2/0 CU Strand
#4 CU Solid
#4 CU Solid
3/0 AL Spacer Cable
2/0 CU Strand
3/0 AL Spacer Cable
#4 CU Solid
#4 CU Solid
#4 CU Solid
3/0 AL Spacer Cable
#4 CU Solid
3/0 AL Spacer Cable
#2 AL Spacer Cable
#2 AL Spacer Cable
#2 AL Spacer Cable
2/0 CU Strand
#2 AL Spacer Cable
#2 AL Spacer Cable
2/0 CU Strand
3/0 AL Spacer Cable
#2 CU Strand
3/0 AL Spacer Cable
3/0 AL Spacer Cable
#2 CU Strand
3/0 AL Spacer Cable
3/0 AL Spacer Cable
#2 CU Strand
3/0 AL Spacer Cable
3/0 AL Spacer Cable
3/0 AL Spacer Cable
#2 CU Strand
3/0 AL Spacer Cable
#4 CU Solid
#4 CU Solid
3/0 AL Spacer Cable
3/0 AL Spacer Cable
#2 CU Strand
3/0 AL Spacer Cable


In [25]:
wires

,shape_wkb,ENABLED,OH_UG,SUBTYPECODE,FEEDERID,OPERVOLT,LOCATIONID,LENGTH,WIRESIZE,GISID,PARENTID,UPDN_PROBLEM,NETWORK_PROBLEM,PHASE,geometry,SOURCETABLE,CALC_CURRENT
0,b'\x01\x02\x00\x00\x00\x02\x00\x00\x00\x00\xb3...,1.0,OH,1.0,TEDESCO,80.0,Glendale Rd,9.031308,2/0 CU Strand,PL00001,PL02103,CHILDLESS,DOWNSTREAM_ONLY,SINGLE,"LINESTRING (824117.666 3003201.359, 824112.042...",PRIMARY,2741.883097
1,b'\x01\x02\x00\x00\x00\x02\x00\x00\x00\x00\xb3...,1.0,OH,3.0,TEDESCO,110.0,Glendale Rd,658.000807,3/0 AL Spacer Cable,PL00002,PL02103,GOOD,GOOD,THREE,"LINESTRING (824117.666 3003201.359, 823615.264...",PRIMARY,45.838006
2,b'\x01\x02\x00\x00\x00\x02\x00\x00\x00\x00K\x1...,1.0,OH,1.0,TEDESCO,80.0,Glendale Rd,8.078712,2/0 CU Strand,PL00003,PL00002,CHILDLESS,DOWNSTREAM_ONLY,SINGLE,"LINESTRING (823615.264 3003626.279, 823607.219...",PRIMARY,3065.190413
3,b'\x01\x02\x00\x00\x00\x03\x00\x00\x00\x00K\x1...,1.0,OH,3.0,TEDESCO,110.0,Glendale Rd,369.999110,3/0 AL Spacer Cable,PL00004,PL00002,GOOD,GOOD,THREE,"LINESTRING (823615.264 3003626.279, 823408.815...",PRIMARY,81.517615
4,b'\x01\x02\x00\x00\x00\x02\x00\x00\x00\x80\x98...,1.0,OH,3.0,TEDESCO,110.0,Humphrey St,67.941158,3/0 AL Spacer Cable,PL00005,SW00111,GOOD,DOWNSTREAM_ONLY,THREE,"LINESTRING (823365.378 3003890.839, 823421.253...",PRIMARY,443.934806
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11309,"b""\x01\x02\x00\x00\x00\x02\x00\x00\x00\x00?*8\...",1.0,UG,1.0,NaN,NaN,NaN,NaN,NaN,SL08662,NaN,PARENTLESS,ISOLATED,None,"LINESTRING (826109.61 3003592.312, 826036.308 ...",SECONDARY,NaN
11310,b'\x01\x02\x00\x00\x00\x02\x00\x00\x00\x00@G\x...,1.0,UG,1.0,NaN,NaN,NaN,NaN,NaN,SL08663,NaN,PARENTLESS,ISOLATED,None,"LINESTRING (826137.559 3003525.867, 826083.242...",SECONDARY,NaN
11311,b'\x01\x02\x00\x00\x00\x02\x00\x00\x00\x00\x0e...,1.0,UG,1.0,NaN,NaN,NaN,NaN,NaN,SL08664,NaN,PARENTLESS,ISOLATED,None,"LINESTRING (835165.26 3006589.645, 835113.602 ...",SECONDARY,NaN
11312,"b""\x01\x02\x00\x00\x00\x02\x00\x00\x00\x00\x8c...",1.0,UG,1.0,NaN,NaN,NaN,NaN,NaN,SL08665,SLU00705,BAD_UPSTREAMID,ISOLATED,None,"LINESTRING (835113.602 3006583.758, 835196.105...",SECONDARY,NaN


In [ ]:
# Plot heat map based on 

## Playground

### Name Parsing

In [ ]:
# Clean and organize wire sizes
references = [
    # --- COPPER ---
    "#6 Copper", "#4 Copper", "#3 Copper", "#2 Copper", "#1 Copper",
    "1/0 Copper", "2/0 Copper", "3/0 Copper", "4/0 Copper",
    "250 kcmil Copper", "300 kcmil Copper", "336 kcmil Copper", 
    "350 kcmil Copper", "400 kcmil Copper", "500 kcmil Copper", 
    "600 kcmil Copper", "750 kcmil Copper", "795 kcmil Copper",
    
    # --- ALUMINUM ---
    "#6 Aluminum", "#4 Aluminum", "#3 Aluminum", "#2 Aluminum", "#1 Aluminum",
    "1/0 Aluminum", "2/0 Aluminum", "3/0 Aluminum", "4/0 Aluminum",
    "250 kcmil Aluminum", "300 kcmil Aluminum", "336 kcmil Aluminum",
    "350 kcmil Aluminum", "400 kcmil Aluminum", "500 kcmil Aluminum", 
    "600 kcmil Aluminum", "750 kcmil Aluminum", "795 kcmil Aluminum" 
]   

In [ ]:
import re

def get_ref_categories(references):
# Create a lookup dictionary for gauges
    gauge_index = {}
    for ref in references:
        #print(f"Ref: {ref}")
        # Regex to find standard wire gauges at the start of the string
        # Matches: 2/0, 1/0, #4, 336, 500, etc.
        match = re.match(r"(?P<gauge>\d+/\d+|#?\d+)", ref.replace("#", ""))
        if match:
            #print(f"Match: {match}")
            gauge = match.group("gauge").replace("#", "") # Normalize: remove #
            #print(f"Gauge: {gauge}")
            if gauge not in gauge_index:
                gauge_index[gauge] = []
                #print(f"New gauge index: {gauge}") 
            #else:
                #print(f"Pre-existing gauge index: {gauge}")  
            gauge_index[gauge].append(ref)
            #print(f"Gauge Index Pair: {gauge}: {gauge_index[gauge]}")
    #print(gauge_index)
    return gauge_index

def smart_wire_match(messy_input, gauge_index):

    if not isinstance(messy_input, str):
        return None
    
    # 1. Extract gauge from messy input
    # Normalize input: remove # for comparison
    input_clean = messy_input.replace("#", "")
    match = re.search(r"(\d+/\d+|\d+)", input_clean)

    # DEBUG
    if match:
        print(f"Input: {messy_input} -> Found Gauge: {match.group(1)}")
    else:
        print(f"Input: {messy_input} -> NO MATCH FOUND")

    if not match:
        return None # No number found, cannot match safely
    
    input_gauge = match.group()

    # 2. Check if this gauge exists in reference index
    if input_gauge not in gauge_index:
        return None # Unknown size
    
    # 3. Fuzzy match ONLY within the correct gauge group
    candidates = gauge_index[input_gauge]

    # Match just the material part (e.g. "Alem" vs "Aluminum")
    best_match, score = process.extractOne(messy_input, candidates)

    # Set a lower threshold since we've guaranteed the size is correct
    if score > 50:
        return best_match
    return None

In [ ]:
### WIP
def material_override(original_input, matched_result):
    input_upper = original_input.upper()

    # Check if a material is specified
    has_material = any(x in input_upper for x in ["COPPER", "CU", "ALUMINUM", "ALUM", "AL"])

    if has_material:
        return matched_result # Trust the result, the material is provided with enough certainty

    # Override if NO material was specified and keyword implies aluminum 
    if any(x in input_upper for x in ["SPACER", "AERIAL", "AREIAL", "PACKAGE"])

In [ ]:
messy_sizes = ['2/0 Copper Strand',
 '3/0 Areial Spacer',
 '#2 Copper Strand',
 '#4 Solid Copper',
 '#2 Areial Spacer',
 '#6 Solid Copper',
 '2/0 Areial Spacer',
 'Package Cable',
 'Spacer Cable',
 '#1 Copper Strand',
 '#4 CU',
 '#2 package cable',
 '1/0 Areial Spacer',
 '1/0 Copper Strand',
 '336 Aerial Spacer',
 '2/0 Copper',
 '#2 Package Cable',
 '336 Spacer Cable',
 '336 spacer Cable',
 '336 Areial Spacer',
 '795 Areial Spacer',
 '3/0 AERIAL SPACER',
 '2 CU',
 '3/0 ALUM',
 '3/0 A;UM',
 '#4 5Kv',
 '2/0 CU',
 '#2 URD',
 '2 ALUM URD',
 '500 MCM CU',
 '#1 EPR CU',
 '2 URD',
 '2',
 '500 mcm cu',
 '2 AL',
 '2/0 AL',
 '#4 AL 4Wire',
 '2/0 AL 3Wire',
 '#2 AL 3Wire',
 '#2 AL',
 '#6 AL 3Wire',
 '#1 CU 3Wire',
 '#6 Copper Strand 3Wi',
 '#4 Copper Strand 3Wi',
 '2/0 AL 4Wire',
 '336 AL 3Wire',
 '#2 AL 4Wire',
 '#2 AL 2Wire',
 '4/0 AL 4Wire',
 '336 AL 4Wire',
 '2 ALUM',
 '2/0 ALUM',
 '2 ALUMINIM',
 '#2 ALUM',
 '#1 CU',
 '#1 CU GROUND',
 '2/0 ALEM',
 '#4 CU GROUND',
 '#6 CU',
 '#2 alum',
 '#6 AL',
 '336 AL',
 '4/0 AL',
 '4/0 CU',
 '350 CU',
 '300 CU',
 '500 CU',
 '350 cu',
 '3/0 CU',
 '500MCM CU',
 '350 MCM',
 '350CU',
 '2/0 Cu',
 '4/0 Al',
 '2 Cu',
 '500 mcm']
messy_sizes

In [ ]:
# try to match wire names with proper ones
for wire in messy_sizes:
    result = smart_wire_match(wire)
    print(f"{wire:<20} -> {result}")

Input: 2/0 Copper Strand -> Found Gauge: 2/0
2/0 Copper Strand    -> 2/0 Copper
Input: 3/0 Areial Spacer -> Found Gauge: 3/0
3/0 Areial Spacer    -> 3/0 Copper
Input: #2 Copper Strand -> Found Gauge: 2
#2 Copper Strand     -> #2 Copper
Input: #4 Solid Copper -> Found Gauge: 4
#4 Solid Copper      -> #4 Copper
Input: #2 Areial Spacer -> Found Gauge: 2
#2 Areial Spacer     -> #2 Copper
Input: #6 Solid Copper -> Found Gauge: 6
#6 Solid Copper      -> #6 Copper
Input: 2/0 Areial Spacer -> Found Gauge: 2/0
2/0 Areial Spacer    -> 2/0 Copper
Input: Package Cable -> NO MATCH FOUND
Package Cable        -> None
Input: Spacer Cable -> NO MATCH FOUND
Spacer Cable         -> None
Input: #1 Copper Strand -> Found Gauge: 1
#1 Copper Strand     -> #1 Copper
Input: #4 CU -> Found Gauge: 4
#4 CU                -> #4 Copper
Input: #2 package cable -> Found Gauge: 2
#2 package cable     -> #2 Copper
Input: 1/0 Areial Spacer -> Found Gauge: 1/0
1/0 Areial Spacer    -> 1/0 Copper
Input: 1/0 Copper Strand -

In [108]:
# how many wires have uncertain names?
weird_names = []
words = ["areial", "spacer", "package", "aerial", "kv", "urd", "epr"]
materials = ["copper", "cu", "aluminum", "al", "a;um", "alum"]

include_pattern = "|".join(re.escape(w) for w in words)
exclude_pattern = "|".join(re.escape(m) for m in materials)

# Condition A: Contains at least one target word (case-insensitive)
has_target = wires["WIRESIZE"].str.contains(include_pattern, case=False, na=False)

# Condition B: Does NOT contain any material word
no_material = ~wires["WIRESIZE"].str.contains(exclude_pattern, case=False, na=False)

# Combine: Must have target AND not have material
weird_wires = wires[has_target & no_material]

print(f"{len(weird_wires)} out of {len(wires)} are weird")
print(weird_wires["WIRESIZE"].unique())

20 out of 11312 are weird
<StringArray>
[   'Package Cable',     'Spacer Cable', '#2 package cable',
 '#2 Package Cable', '336 Spacer Cable', '336 spacer Cable',
           '#4 5Kv',           '#2 URD',            '2 URD']
Length: 9, dtype: str


In [109]:
# how many wires have no gauge?
wires["HASGAUGE"] = wires["WIRESIZE"].apply(lambda x: any(char.isdigit() for char in str(x)))
print(f"{len(wires[~wires["HASGAUGE"]])} out of {len(wires)} do not have a gauge value")

1065 out of 11312 do not have a gauge value


In [92]:
# fix aerial (areial) typo
wires["WIRESIZE"] = wires["WIRESIZE"].str.replace("Areial", "Aerial", regex=True)